# Phase 1b -- Content-Emotion Analysis of the Source Dataset

**PersonaPlex in-house emotion network project** -- read-only analysis of the synthetic user-voice dataset (~10,000 clips). It answers one question Phase 1 could not answer without the pod: **how many clips have emotionally *charged words* (content), per emotion, and how does that intersect with the spoken (prosody) emotion and the closed-loop QA verdicts?**

Why this matters: Phase 1 showed PersonaPlex mirrors the **content** of the user turn, not its prosody. The supplementary collection round (`STEERING_MODE=content`) draws clips by *content* emotion -- this analysis measures exactly how many usable clips exist per emotion (especially **fear** and **sad**, the starved classes) before committing GPU hours.

## What this notebook does

1. Locates the manifest + QA file + WAV directory on the pod (reports the actual paths, file counts, total size).
2. Replicates Phase 1's pool-building + QA merge logic 1:1 (numbers reconcile with Phase 1 cell 12: 10,000 clips, 8,622 QA-verified).
3. Scores every clip's **text** with the same text teacher Phase 1 uses (`j-hartmann/emotion-english-distilroberta-base`; disgust folds into angry, DISGUST_MAP=3) -> content emotion + confidence per clip.
4. Prints the content-emotion distribution at confidence thresholds 0.5/0.6/0.7 with QA-verified/non-QA splits, the prosody x content cross-matrix, and a fear/sad deep-dive.
5. Writes report files and a rule-based recommendation for the supplementary round.

## Read-only guarantee

No WAV, manifest, QA, session, or window file is modified. Only three **new** files are written (see Outputs). No PersonaPlex engine is needed -- only the text teacher (~1 GB VRAM).

## Outputs

| File | Where | Purpose |
|---|---|---|
| `content_emotion_analysis.csv` | `/workspace/persona_emotion/phase1/` | per-clip detail (prosody, QA, content, all 6 scores) |
| `content_emotion_report.json` | `/workspace/persona_emotion/phase1/` | machine-readable summary for the config decision |
| `clip_content_emotions.csv` | `/workspace/persona_emotion/phase1/assets/` | steering cache -- the Phase 1 content-steering cell loads this instead of re-scoring |

Runtime: ~5-10 min (text teacher on 10k short texts = ~2-4 min on GPU).

In [ ]:
# ============ Configuration + locate the source dataset on this pod =========
import os, sys, json, time, subprocess
from pathlib import Path

WORKSPACE = Path(os.environ.get("WORKSPACE", "/workspace"))
PHASE1_DIR = Path(os.environ.get("PERSONA_EMO_PHASE1_DIR", str(WORKSPACE / "persona_emotion" / "phase1")))
ASSETS_DIR = PHASE1_DIR / "assets"
PHASE1_DIR.mkdir(parents=True, exist_ok=True); ASSETS_DIR.mkdir(parents=True, exist_ok=True)

EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
EMOTION_IDS = {v: k for k, v in EMOTION_NAMES.items()}
DISGUST_MAP = 3
TEXT_TEACHER_MODEL = "j-hartmann/emotion-english-distilroberta-base"

# numpy/pandas ABI guard (same as Phase 1): pandas 2.1.x + numpy 2.x on disk crashes on import
_r = subprocess.run([sys.executable, "-c", "import numpy, pandas"], capture_output=True, text=True)
if _r.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall",
                    "numpy<2.0.0", "pandas<2.2.0"], capture_output=True)
    raise RuntimeError("numpy/pandas ABI mismatch repaired on disk (numpy<2 restored). RESTART THE KERNEL, then re-run all cells.")

def log(msg):
    print("[%s] %s" % (time.strftime("%H:%M:%S"), msg), flush=True)

# locate the source dataset: env override first, then the two known pod locations
_cands = []
if os.environ.get("USER_VOICE_DATASET_DIR"):
    _cands.append(Path(os.environ["USER_VOICE_DATASET_DIR"]))
_cands += [WORKSPACE / "voice_dataset", WORKSPACE]
MANIFEST = None
for _d in _cands:
    if (_d / "synthetic_dataset_manifest.csv").exists():
        MANIFEST = _d / "synthetic_dataset_manifest.csv"
        DATASET_DIR = _d
        break
if MANIFEST is None:
    raise RuntimeError("synthetic_dataset_manifest.csv not found in: %s" % [str(c) for c in _cands])
QA_CSV = DATASET_DIR / "dataset_emotion_qa.csv"
log("manifest : %s" % MANIFEST)
log("QA file  : %s (exists=%s)" % (QA_CSV, QA_CSV.exists()))

In [ ]:
# ============ Resolve WAV files + prosody pools + QA merge ==================
# 1:1 replication of Phase 1's load_user_voice_dataset logic, so the numbers
# reconcile with Phase 1 cell 12 (10,000 clips, 8,622 QA-verified).
import pandas as pd
df = pd.read_csv(MANIFEST)
required = {"id", "text", "emotion", "audio_path"}
missing_cols = required - set(df.columns)
if missing_cols:
    raise RuntimeError("manifest %s is missing columns: %s" % (MANIFEST, sorted(missing_cols)))
paths, n_missing = [], 0
for row in df.itertuples(index=False):
    p = Path(str(row.audio_path))
    if not p.is_absolute():
        p = DATASET_DIR / p
    if not p.exists():
        alt = DATASET_DIR / "synthetic_user_clips" / Path(str(row.audio_path)).name
        p = alt if alt.exists() else p
    if not p.exists():
        n_missing += 1
        paths.append(None)
    else:
        paths.append(p)
found = [p for p in paths if p is not None]
wav_dirs = {}
for p in found:
    wav_dirs[str(p.parent)] = wav_dirs.get(str(p.parent), 0) + 1
_total_gb = sum(p.stat().st_size for p in found) / 1e9
log("WAV files: %d found | %d missing | %.2f GB total" % (len(found), n_missing, _total_gb))
log("WAV directories: %s" % json.dumps(wav_dirs))

clips = []
for row, p in zip(df.itertuples(index=False), paths):
    if p is None:
        continue
    emo_raw = str(row.emotion).strip().lower()
    emo6 = "angry" if emo_raw == "disgust" else emo_raw
    if emo6 not in EMOTION_IDS:
        continue
    clips.append({"clip_id": str(row.id), "text": str(row.text), "emotion_raw": emo_raw,
                  "emotion": emo6, "qa_match": None, "qa_predicted": ""})
log("clips resolved: %d" % len(clips))
if QA_CSV.exists():
    qa = pd.read_csv(QA_CSV)
    _qa = {str(r.id): (bool(r.match), str(getattr(r, "predicted", "")).strip().lower())
           for r in qa.itertuples(index=False)}
    for c in clips:
        if c["clip_id"] in _qa:
            _m, _pred = _qa[c["clip_id"]]
            c["qa_match"] = bool(_m or (c["emotion_raw"] == "disgust" and _pred in ("disgust", "angry")))
            c["qa_predicted"] = _pred
else:
    log("WARNING: no QA file -- qa_match counts will be 0")
log("QA-verified: %d / %d" % (sum(1 for c in clips if c["qa_match"]), len(clips)))
for emo in EMOTION_NAMES.values():
    lst = [c for c in clips if c["emotion"] == emo]
    log("prosody pool %-8s: %5d clips (%d QA-verified)" % (emo, len(lst), sum(1 for c in lst if c["qa_match"])))

In [ ]:
# ============ Content-emotion scoring (text teacher, same model as Phase 1) ===
import numpy as np, torch
from transformers import pipeline as hf_pipeline
TEXT_TO_ID = {"anger": 3, "joy": 1, "neutral": 0, "sadness": 2, "surprise": 4, "fear": 5, "disgust": DISGUST_MAP}
_pipe = hf_pipeline("text-classification", model=TEXT_TEACHER_MODEL, top_k=None,
                    device=0 if torch.cuda.is_available() else -1)
log("scoring %d clip texts (content emotion)..." % len(clips))
texts = [c["text"] for c in clips]
_B = 64
for _i in range(0, len(texts), _B):
    for c, out in zip(clips[_i:_i + _B], _pipe([t[:512] for t in texts[_i:_i + _B]])):
        entries = out if isinstance(out, list) else [out]
        v = np.zeros(6)
        for r in entries:
            eid = TEXT_TO_ID.get(str(r["label"]).lower())
            if eid is not None:
                v[eid] = max(v[eid], float(r["score"]))
        s = float(v.sum())
        if s > 0:
            v = v / s
        cid = int(v.argmax())
        c["content_emotion"] = EMOTION_NAMES[cid]
        c["content_conf"] = round(float(v[cid]), 4)
        c["content_scores"] = [round(float(x), 4) for x in v]
    if (_i // _B) % 20 == 0:
        log("  scored %d/%d" % (min(_i + _B, len(texts)), len(texts)))

detail = pd.DataFrame([{
    "clip_id": c["clip_id"], "prosody_emotion_raw": c["emotion_raw"], "prosody_emotion": c["emotion"],
    "qa_match": c["qa_match"], "qa_predicted": c["qa_predicted"],
    "content_emotion": c["content_emotion"], "content_conf": c["content_conf"],
    **{"score_" + e: c["content_scores"][k] for k, e in EMOTION_NAMES.items()}
} for c in clips])
ANALYSIS_CSV = PHASE1_DIR / "content_emotion_analysis.csv"
detail.to_csv(ANALYSIS_CSV, index=False)
log("per-clip detail -> %s (%d rows)" % (ANALYSIS_CSV, len(detail)))
CACHE_CSV = ASSETS_DIR / "clip_content_emotions.csv"
detail[["clip_id", "content_emotion", "content_conf"]].to_csv(CACHE_CSV, index=False)
log("steering cache -> %s (the Phase 1 content-steering cell loads this instead of re-scoring)" % CACHE_CSV)

In [ ]:
# ============ Content-emotion distribution report (printed) =================
print("=== content emotion distribution (all clips) ===")
print(detail.content_emotion.value_counts().to_string())
for thr in (0.5, 0.6, 0.7):
    sel = detail[detail.content_conf >= thr]
    print()
    print("=== content pools @ conf >= %.1f (QA-verified / non-QA) ===" % thr)
    for emo in EMOTION_NAMES.values():
        s = sel[sel.content_emotion == emo]
        nv = int(s.qa_match.fillna(False).sum())
        print("  %-8s: %5d total (%d QA-verified / %d non-QA)" % (emo, len(s), nv, len(s) - nv))
print()
print("=== prosody (rows) x content (cols) cross-matrix ===")
print(pd.crosstab(detail.prosody_emotion, detail.content_emotion).to_string())
print()
print("=== FEAR deep-dive ===")
f = detail[detail.content_emotion == "fear"]
print("content=fear: %d clips (conf>=0.6: %d | QA-verified@0.6: %d)" % (
    len(f), int((f.content_conf >= 0.6).sum()),
    int(f[f.content_conf >= 0.6].qa_match.fillna(False).sum())))
print("double-fear (prosody=fear AND content=fear): %d" % int((detail.prosody_emotion.eq("fear") & detail.content_emotion.eq("fear")).sum()))
print("of prosody=fear clips (%d), content is: %s" % (
    int(detail.prosody_emotion.eq("fear").sum()),
    detail[detail.prosody_emotion == "fear"].content_emotion.value_counts().to_dict()))
print()
print("=== SAD deep-dive ===")
sd = detail[detail.content_emotion == "sad"]
print("content=sad: %d clips (conf>=0.6: %d | QA-verified@0.6: %d)" % (
    len(sd), int((sd.content_conf >= 0.6).sum()),
    int(sd[sd.content_conf >= 0.6].qa_match.fillna(False).sum())))
print("double-sad (prosody=sad AND content=sad): %d" % int((detail.prosody_emotion.eq("sad") & detail.content_emotion.eq("sad")).sum()))
print("of prosody=sad clips (%d), content is: %s" % (
    int(detail.prosody_emotion.eq("sad").sum()),
    detail[detail.prosody_emotion == "sad"].content_emotion.value_counts().to_dict()))

In [ ]:
# ============ JSON report + supplementary-collection recommendation =========
REPORT_JSON = PHASE1_DIR / "content_emotion_report.json"

def _pools(thr):
    out = {}
    for emo in EMOTION_NAMES.values():
        s = detail[(detail.content_emotion == emo) & (detail.content_conf >= thr)]
        out[emo] = {"total": int(len(s)), "qa_verified": int(s.qa_match.fillna(False).sum())}
    return out

report = {
    "analysis": "content-emotion distribution of the synthetic user-voice dataset (Phase 1b)",
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "manifest": str(MANIFEST), "wav_directories": wav_dirs,
    "wav_files_found": len(found), "wav_files_missing": n_missing, "wav_total_gb": round(_total_gb, 2),
    "clips_total": len(clips), "text_teacher": TEXT_TEACHER_MODEL,
    "prosody_pools": {emo: {"total": int(detail.prosody_emotion.eq(emo).sum()),
                            "qa_verified": int(detail[detail.prosody_emotion == emo].qa_match.fillna(False).sum())}
                      for emo in EMOTION_NAMES.values()},
    "content_distribution_all": {k: int(v) for k, v in detail.content_emotion.value_counts().items()},
    "content_pools_by_threshold": {"0.5": _pools(0.5), "0.6": _pools(0.6), "0.7": _pools(0.7)},
    "prosody_x_content": {k: {kk: int(vv) for kk, vv in v.items()}
                          for k, v in pd.crosstab(detail.prosody_emotion, detail.content_emotion).to_dict().items()},
    "fear": {"content_total": int(detail.content_emotion.eq("fear").sum()),
             "content_conf_ge_0.6": int((detail.content_emotion.eq("fear") & (detail.content_conf >= 0.6)).sum()),
             "double_fear": int((detail.prosody_emotion.eq("fear") & detail.content_emotion.eq("fear")).sum())},
    "sad": {"content_total": int(detail.content_emotion.eq("sad").sum()),
            "content_conf_ge_0.6": int((detail.content_emotion.eq("sad") & (detail.content_conf >= 0.6)).sum()),
            "double_sad": int((detail.prosody_emotion.eq("sad") & detail.content_emotion.eq("sad")).sum())},
    "files": {"detail_csv": str(ANALYSIS_CSV), "steering_cache": str(CACHE_CSV),
              "report_json": str(REPORT_JSON)},
}
json.dump(report, open(REPORT_JSON, "w"), indent=2)
log("report -> %s" % REPORT_JSON)

# rule-based recommendation (final config call is made after reviewing this report)
for emo in ("fear", "sad"):
    p6 = report["content_pools_by_threshold"]["0.6"][emo]["total"]
    p5 = report["content_pools_by_threshold"]["0.5"][emo]["total"]
    if p6 >= 160:
        log("recommendation: %-4s content pool @0.6 = %d -> CONTENT_CONF=0.6 is fine for a 960-session round" % (emo, p6))
    elif p5 >= 160:
        log("recommendation: %-4s content pool @0.6 = %d but @0.5 = %d -> use CONTENT_CONF=0.5" % (emo, p6, p5))
    else:
        log("recommendation: %-4s content pool is thin (@0.6=%d, @0.5=%d) -> content steering alone will not cover %s; consider generating new %s-content clips" % (emo, p6, p5, emo, emo))
_ceiling = 6 * min(v["total"] for v in report["content_pools_by_threshold"]["0.6"].values())
log("content-mode no-reuse ceiling @0.6: ~%d sessions (6 x smallest pool)" % _ceiling)
log("Phase 1b analysis complete. Download content_emotion_report.json + content_emotion_analysis.csv (NOT the WAVs).")

## Using the results

1. **Download only** `content_emotion_report.json` (~5 KB) and `content_emotion_analysis.csv` (~1.5 MB) to your PC -- never the WAVs.
2. Reconcile the numbers with Phase 1 (10,000 clips; 8,622 QA-verified; prosody pools 1428/1429/1429/2857/1429/1428) -- any mismatch means the dataset state drifted and should be investigated before collection.
3. **Supplementary round config** (set before the Phase 1 content cell):
   - `os.environ["STEERING_MODE"] = "content"` -- the steering cell loads `assets/clip_content_emotions.csv` (this notebook already wrote it) and rebuilds the pools by content emotion.
   - `CONTENT_CONF` per the recommendation above (0.6 if the fear/sad pools @0.6 are large enough, else 0.5).
   - `TARGET_SESSIONS` per the recommendation -- capped by the printed content-mode no-reuse ceiling (6 x smallest content pool) to avoid clip duplication.
4. Then re-run Phase 1's collection cell (resume-safe: the existing 240 sessions are kept).